# NLP Spam Sentiment Classification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

from collections import Counter
from wordcloud import WordCloud, STOPWORDS


path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
print("Dataset downloaded to:", path)



# Load CSV, check shape and dtypes

In [ ]:
df = pd.read_csv(f"{path}/spam.csv", encoding='latin1')
df.head()

In [ ]:
df.tail(5)

# Summary Statistics

In [ ]:
df.info()
print(df.shape)
df.describe()

# Data Cleaning and Preprocessing



## Dropping Unnecessary Columns



In [ ]:
df = df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)


### Renaming Columns for Clarity

In [ ]:
df = df.rename(columns={'v1':'label', 'v2':'text'})
df.head()

## Checking for Missing Values



In [ ]:
df.isnull().sum()

##  Helper Functions for Exploratory Data Analysis



In [190]:
def check_class_balance(df, target_column):
    counts = df[target_column].value_counts()
    percentages = df[target_column].value_counts(normalize=True) * 100

    print("Counts:\n", counts)
    print("\nPercentages:\n", percentages)

    spam_pct = (df[target_column] == "spam").mean() * 100
    if 12 <= spam_pct <= 13:
        print("\nNote: Dataset is imbalanced (~12-13% spam)")


def check_text_length(df, text_column, label_column):
    df["text_length"] = df[text_column].astype(str).apply(len)
    print(df.groupby(label_column)["text_length"].mean())


def show_top_words(df, text_col, label_col, top_n=10, stopwords=None):
    results = {}
    for category in df[label_col].unique():
        all_text = " ".join(df[df[label_col] == category][text_col].astype(str))
        words = [word.lower() for word in all_text.split() if word.lower().isalpha()]
        if stopwords:
            words = [word for word in words if word not in stopwords]
        common = Counter(words).most_common(top_n)
        print(f"\nTop {top_n} words in {category}:", common)
        results[category] = common
    return results

In [202]:
custom_stopwords = set(list(STOPWORDS) + ['u', 'ur', '2', '4', 'im', 'dont', 'do', 'will', 'go', 'got', 'like', 'come', 'know', 'get'])
check_class_balance(df, "label")
check_text_length(df, "text", "label")

show_top_words(df, "text", "label", top_n=10, stopwords=custom_stopwords)

In [203]:
word_distributions = show_top_words(df, 'text', 'label', top_n=10, stopwords=custom_stopwords)

ham_top_words = word_distributions.get('ham', [])
spam_top_words = word_distributions.get('spam', [])

ham_words = " ".join(df[df['label'] == 'ham']['text'].astype(str))
spam_words = " ".join(df[df['label'] == 'spam']['text'].astype(str))

ham_df = pd.DataFrame(ham_top_words, columns=['word', 'count'])
spam_df = pd.DataFrame(spam_top_words, columns=['word', 'count'])

ham_wordcloud = WordCloud(width=800, height=400, background_color='white', stopwords=STOPWORDS).generate(ham_words)
spam_wordcloud = WordCloud(width=800, height=400, background_color='white', stopwords=STOPWORDS).generate(spam_words)

# Exploratory Data Analysis

In [ ]:
sns.set_style('whitegrid')
palette = ['#4C72B0', '#DD8452']
model_palette = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#937860', '#E377C2']
metric_palette = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(x='label', data=df, hue='label', palette=palette, legend=False, edgecolor='black')
plt.title('Distribution of Ham vs. Spam Messages')
plt.xlabel('Message Type')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='text_length', hue='label', bins=50, palette=palette, kde=True)
plt.title('Text Length Distribution by Message Type')
plt.xlabel('Text Length')
plt.ylabel('Frequency')
plt.show()

In [ ]:

plt.figure(figsize=(10, 5))
sns.barplot(x='count', y='word', data=ham_df, palette='Blues_r', hue='word', legend=False)
plt.title('Top 10 Most Common Words in Ham Messages')
plt.xlabel('Count')
plt.ylabel('Words')
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x='count', y='word', data=spam_df, palette='Oranges_r', hue='word', legend=False)
plt.title('Top 10 Most Common Words in Spam Messages')
plt.xlabel('Count')
plt.ylabel('Words')
plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(10, 5))
plt.imshow(ham_wordcloud, interpolation='bilinear')
plt.title('Ham Messages Word Cloud', fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(spam_wordcloud, interpolation='bilinear')
plt.title('Spam Messages Word Cloud', fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()